# Customer Segmentation with K-Means (RFM + Behavioural Features)

**Dataset:** [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail) — all transactions between 01/12/2010 and 09/12/2011 for a UK-based online retailer.

**Goal:** group customers into actionable segments based on *how recently, how often, and how much* they buy, plus basket-level behaviour.

**Pipeline**

| # | Section | What happens |
|---|---------|--------------|
| 1 | Setup | Imports, plotting style, global config |
| 2 | Load data | Read the CSV, parse dates |
| 3 | Exploratory analysis | Data quality, distributions, trends, countries |
| 4 | Data cleaning | Remove unusable rows, with an audit trail |
| 5 | Feature engineering | One row per customer: RFM + behavioural features |
| 6 | Preprocessing | Outlier capping, log transform, scaling |
| 7 | Choosing K | Elbow, silhouette, Davies-Bouldin, Calinski-Harabasz |
| 8 | Final model & PCA view | Train K-Means, visualise in 2-D |
| 9 | Segment profiling | Who is in each cluster, and how much revenue they bring |
| 10 | Export | Save the segment assignments |

---
## 1. Setup

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

# ---- Global configuration -------------------------------------------------
RANDOM_STATE = 42          # one seed used everywhere for reproducibility
DATA_URL = "https://archive.ics.uci.edu/static/public/352/data.csv"
OUTLIER_QUANTILE = 0.99    # upper cap applied to every clustering feature
K_RANGE = range(2, 11)     # candidate numbers of clusters
MIN_K = 3                  # see section 7 for why K=2 is excluded

print("Setup complete.")

---
## 2. Load Data

In [ ]:
df = pd.read_csv(DATA_URL, encoding="ISO-8859-1")

# Parse dates once, up front. Left as strings, min()/max() would sort alphabetically.
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format="%m/%d/%Y %H:%M")

print("Shape:", df.shape)
display(df.head())
df.info()

---
## 3. Exploratory Data Analysis

### 3.1 Data quality overview

In [ ]:
display(df.describe(include="all").T)

print("Duplicate rows       :", df.duplicated().sum())
print("Unique customers     :", df["CustomerID"].nunique())
print("Date range           :", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

cancelled = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")
print(f"Cancelled invoices   : {cancelled.sum():,} rows ({cancelled.mean():.1%})")

In [ ]:
missing = (df.isnull().mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=missing.values, y=missing.index, color="steelblue", ax=ax)
ax.set_title("Missing Values by Column")
ax.set_xlabel("Missing (%)")
ax.set_ylabel("")
for i, v in enumerate(missing.values):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center")
plt.tight_layout()
plt.show()

**Takeaway:** about a quarter of rows have no `CustomerID`. They cannot be attributed to a customer, so they are dropped in the cleaning step.

### 3.2 Quantity and price distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df["Quantity"].clip(-100, 100), bins=50, kde=True, ax=axes[0], color="royalblue")
axes[0].set_title("Quantity Distribution (clipped to ±100)")

sns.histplot(df["UnitPrice"].clip(upper=100), bins=50, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Unit Price Distribution (clipped at 100)")

plt.tight_layout()
plt.show()

Both variables are heavily right-skewed and contain negative or zero values (returns, adjustments), which also motivates the cleaning rules below.

### 3.3 Monthly trends (cancelled invoices excluded)

In [ ]:
valid = df[~df["InvoiceNo"].astype(str).str.upper().str.startswith("C")].copy()
valid["Month"] = valid["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
valid["Revenue"] = valid["Quantity"] * valid["UnitPrice"]

monthly = valid.groupby("Month").agg(
    Invoices=("InvoiceNo", "nunique"),
    Revenue=("Revenue", "sum"),
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(monthly.index, monthly["Invoices"], marker="o", color="teal", linewidth=2)
axes[0].set_title("Unique Invoices per Month")
axes[0].set_ylabel("Invoices")

axes[1].plot(monthly.index, monthly["Revenue"], marker="o", color="firebrick", linewidth=2)
axes[1].set_title("Revenue per Month")
axes[1].set_ylabel("Revenue (£)")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

**Notes:** activity ramps up strongly from September towards the Christmas peak (typical for a gift retailer). The final month (Dec 2011) is **partial**, since the data ends on 9 December, so its drop is not a real decline.

### 3.4 Top countries

In [ ]:
country_counts = df["Country"].value_counts().head(10)
uk_share = (df["Country"] == "United Kingdom").mean()

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(
    x=country_counts.values, y=country_counts.index,
    hue=country_counts.index, palette="viridis", legend=False, ax=ax,
)
ax.set_xscale("log")
ax.set_title("Top 10 Countries by Transaction Records (log scale)")
ax.set_xlabel("Transaction records")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

print(f"United Kingdom share of all records: {uk_share:.1%}")

---
## 4. Data Cleaning

Rules applied, in order:

1. Drop rows without `CustomerID`; segmentation needs a customer.
2. Drop exact duplicate rows.
3. Drop cancelled invoices (`InvoiceNo` starting with `C`).
4. Drop non-positive `Quantity` and `UnitPrice` (returns, adjustments, free items).
5. Drop **non-product stock codes** (postage, manual adjustments, bank charges, fees, ...). Real product codes start with 5 digits; the rest would otherwise inflate spend and basket features.
6. Fix dtypes and create `TotalAmount = Quantity × UnitPrice`.

The log below shows how many rows each rule removes.

In [ ]:
audit = [("Raw data", len(df))]

d = df.dropna(subset=["CustomerID"])
audit.append(("1. Drop missing CustomerID", len(d)))

d = d.drop_duplicates()
audit.append(("2. Drop duplicates", len(d)))

d = d[~d["InvoiceNo"].astype(str).str.upper().str.startswith("C")]
audit.append(("3. Drop cancelled invoices", len(d)))

d = d[(d["Quantity"] > 0) & (d["UnitPrice"] > 0)]
audit.append(("4. Drop non-positive qty/price", len(d)))

d = d[d["StockCode"].astype(str).str.match(r"^\d{5}")]
audit.append(("5. Drop non-product stock codes", len(d)))

df_clean = d.copy()
df_clean["CustomerID"] = df_clean["CustomerID"].astype("int64")
df_clean["TotalAmount"] = df_clean["Quantity"] * df_clean["UnitPrice"]
df_clean["InvoiceMonth"] = df_clean["InvoiceDate"].dt.to_period("M")

audit_df = pd.DataFrame(audit, columns=["Step", "Rows remaining"])
audit_df["Rows removed"] = -audit_df["Rows remaining"].diff().fillna(0).astype(int)
display(audit_df.style.format({"Rows remaining": "{:,}", "Rows removed": "{:,}"}))

print(f"\nKept {len(df_clean):,} of {len(df):,} rows ({len(df_clean) / len(df):.1%})")

In [ ]:
assert df_clean["CustomerID"].notna().all()
assert (df_clean["Quantity"] > 0).all()
assert (df_clean["UnitPrice"] > 0).all()
assert (df_clean["TotalAmount"] > 0).all()
assert not df_clean.duplicated().any()
assert df_clean.isnull().sum().sum() == 0

print("Data validation passed.")
display(df_clean.dtypes.to_frame("dtype").T)

---
## 5. Feature Engineering

One row per customer. Features fall into three groups:

| Group | Feature | Definition |
|-------|---------|------------|
| **RFM** | `Recency` | Days since last purchase (relative to the day after the last transaction in the data) |
| | `Frequency` | Number of distinct invoices |
| | `Monetary` | Total spend |
| **Basket** | `AverageOrderValue` | `Monetary / Frequency` |
| | `AverageBasketSize` | Items per invoice |
| | `TotalQuantity`, `UniqueProducts` | Volume and variety of purchases |
| **Lifecycle** | `ActiveMonths` | Distinct months with at least one purchase |
| | `CustomerTenure` | Days between first and last purchase |

In [ ]:
reference_date = df_clean["InvoiceDate"].max() + pd.Timedelta(days=1)
print("Reference date for Recency:", reference_date.date())

customer_features = (
    df_clean.groupby("CustomerID")
    .agg(
        FirstPurchase=("InvoiceDate", "min"),
        LastPurchase=("InvoiceDate", "max"),
        Monetary=("TotalAmount", "sum"),
        TotalQuantity=("Quantity", "sum"),
        UniqueProducts=("StockCode", "nunique"),
        Frequency=("InvoiceNo", "nunique"),
        ActiveMonths=("InvoiceMonth", "nunique"),
        Country=("Country", "last"),
    )
    .reset_index()
)

customer_features["Recency"] = (reference_date - customer_features["LastPurchase"]).dt.days
customer_features["CustomerTenure"] = (
    customer_features["LastPurchase"] - customer_features["FirstPurchase"]
).dt.days

# Mean of per-invoice sums equals total / number of invoices, so no invoice-level table is needed.
customer_features["AverageOrderValue"] = customer_features["Monetary"] / customer_features["Frequency"]
customer_features["AverageBasketSize"] = customer_features["TotalQuantity"] / customer_features["Frequency"]

print("Customer feature matrix:", customer_features.shape)
display(customer_features.head(10))

### 5.1 Feature quality check

In [ ]:
numeric_features = customer_features.drop(columns="CustomerID").select_dtypes(include=np.number)

print("Missing values:", int(numeric_features.isnull().sum().sum()))
print("Infinite values:", int(np.isinf(numeric_features).sum().sum()))
display(numeric_features.describe().T)

corr = numeric_features.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(11, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, square=True, cbar_kws={"shrink": 0.8})
plt.title("Customer Feature Correlation Matrix")
plt.tight_layout()
plt.show()

**Takeaway:** the volume features (`Monetary`, `TotalQuantity`, `Frequency`, `UniqueProducts`, `ActiveMonths`) are strongly correlated, so the clustering will be driven largely by one "customer size / engagement" axis (visible later as a dominant PC1). That is acceptable for segmentation, but it is worth knowing. If you want the clusters to depend more on *behaviour* than on *size*, try a smaller set such as `Recency`, `Frequency`, `Monetary` and `AverageOrderValue`.

---
## 6. Preprocessing for K-Means

K-Means uses Euclidean distance, so it is sensitive to outliers, skew and feature scale. Three steps:

1. **Cap outliers** at the 99th percentile.
2. **Log-transform** (`log1p`) to reduce right skew.
3. **Standardise** to zero mean and unit variance.

In [ ]:
feature_columns = [
    "Recency", "Frequency", "Monetary", "AverageOrderValue", "TotalQuantity",
    "UniqueProducts", "AverageBasketSize", "ActiveMonths", "CustomerTenure",
]

X = customer_features[feature_columns].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median())

assert X.notna().all().all() and np.isfinite(X.to_numpy()).all()
print("Feature matrix:", X.shape)

In [ ]:
# 6.1 Outlier capping
upper_limits = X.quantile(OUTLIER_QUANTILE)
X_capped = X.clip(upper=upper_limits, axis=1)

display(pd.DataFrame({
    "Original max": X.max(),
    f"p{int(OUTLIER_QUANTILE * 100)} cap": upper_limits,
    "Capped max": X_capped.max(),
    "Customers capped": (X > upper_limits).sum(),
}))

In [ ]:
# 6.2 Log transform
assert (X_capped >= 0).all().all(), "log1p requires non-negative values"
X_log = np.log1p(X_capped)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(X["Monetary"], bins=40, ax=axes[0], color="royalblue")
axes[0].set_title("Monetary: original")
axes[0].set_xlabel("Customer spend (£)")

sns.histplot(X_log["Monetary"], bins=40, ax=axes[1], color="darkorange")
axes[1].set_title("Monetary: capped + log1p")
axes[1].set_xlabel("log(1 + spend)")

for ax in axes:
    ax.set_ylabel("Customers")

plt.tight_layout()
plt.show()

In [ ]:
# 6.3 Standardisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

X_scaled_df = pd.DataFrame(X_scaled, columns=feature_columns, index=customer_features.index)

print("Scaled matrix:", X_scaled.shape)
display(X_scaled_df.agg(["mean", "std"]).round(3))

---
## 7. Choosing the Number of Clusters

Four complementary criteria are computed for each candidate K:

- **Inertia (elbow):** within-cluster sum of squares. Look for where improvement flattens.
- **Silhouette:** higher is better (range −1 to 1).
- **Davies-Bouldin:** lower is better.
- **Calinski-Harabasz:** higher is better.

In [ ]:
results = []

for k in K_RANGE:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = model.fit_predict(X_scaled)
    results.append({
        "K": k,
        "Inertia": model.inertia_,
        "Silhouette": silhouette_score(X_scaled, labels),
        "Davies_Bouldin": davies_bouldin_score(X_scaled, labels),
        "Calinski_Harabasz": calinski_harabasz_score(X_scaled, labels),
    })

evaluation_df = pd.DataFrame(results)
display(evaluation_df.round(4).style.format({"Inertia": "{:,.0f}", "Calinski_Harabasz": "{:,.0f}"}))

In [ ]:
panels = [
    ("Inertia", "Elbow: Inertia (lower, look for the bend)"),
    ("Silhouette", "Silhouette Score (higher is better)"),
    ("Davies_Bouldin", "Davies-Bouldin Index (lower is better)"),
    ("Calinski_Harabasz", "Calinski-Harabasz Index (higher is better)"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, (column, title) in zip(axes.ravel(), panels):
    sns.lineplot(data=evaluation_df, x="K", y=column, marker="o", linewidth=2, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Number of clusters (K)")
    ax.set_xticks(list(K_RANGE))

plt.tight_layout()
plt.show()

### Decision

K = 2 usually wins on the raw metrics here, but it simply splits customers into "active" vs "inactive", which is too coarse to act on. We therefore pick the K with the best silhouette score **among K ≥ 3** (`MIN_K` in the config cell). The elbow plot shows no sharp bend, so this is a judgement call that combines the metrics with business usefulness. Change `MIN_K` or set `FINAL_K` by hand to experiment.

---
## 8. Final Model & PCA Visualisation

### 8.1 Train K-Means

Cluster IDs are re-ordered by average spend, so **Cluster 0 is always the highest-value segment**. The ordering is stable across re-runs.

In [ ]:
candidates = evaluation_df[evaluation_df["K"] >= MIN_K]
FINAL_K = int(candidates.loc[candidates["Silhouette"].idxmax(), "K"])
print("Selected K:", FINAL_K)

kmeans = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=50, max_iter=500)
raw_labels = kmeans.fit_predict(X_scaled)

# Re-label clusters by descending mean spend
order = customer_features["Monetary"].groupby(raw_labels).mean().sort_values(ascending=False).index
relabel = {old: new for new, old in enumerate(order)}
customer_features["Cluster"] = pd.Series(raw_labels).map(relabel).values

CLUSTER_COLORS = sns.color_palette("Set2", FINAL_K)

assert customer_features["Cluster"].notna().all()
assert len(customer_features) == len(X_scaled)

cluster_counts = customer_features["Cluster"].value_counts().sort_index()
display(pd.DataFrame({
    "Customers": cluster_counts,
    "Share (%)": (cluster_counts / len(customer_features) * 100).round(2),
}))

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=cluster_counts.index, y=cluster_counts.values, hue=cluster_counts.index,
            palette=CLUSTER_COLORS, legend=False, ax=ax)
ax.set_title("Customers per Cluster")
ax.set_xlabel("Cluster")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()

### 8.2 Visualise clusters in 2-D with PCA

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["Cluster"] = customer_features["Cluster"].values

var = pca.explained_variance_ratio_ * 100
print(f"PC1: {var[0]:.1f}%   PC2: {var[1]:.1f}%   Total: {var.sum():.1f}% of variance")

fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Cluster", palette=CLUSTER_COLORS,
                s=30, alpha=0.65, edgecolor=None, ax=ax)

centroids = pca_df.groupby("Cluster")[["PC1", "PC2"]].mean()
ax.scatter(centroids["PC1"], centroids["PC2"], c="black", marker="X", s=180, label="Centroid")

ax.set_title("Customer Segments (K-Means, projected with PCA)")
ax.set_xlabel(f"PC1 ({var[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({var[1]:.1f}% variance)")
ax.legend(title="Cluster")
plt.tight_layout()
plt.show()

loadings = pd.DataFrame(pca.components_.T, index=feature_columns, columns=["PC1", "PC2"])
print("\nPCA loadings (what each component represents):")
display(loadings.round(2))

### 8.3 Final cluster quality

In [ ]:
final_labels = customer_features["Cluster"]

display(pd.DataFrame({
    "Metric": ["Number of clusters", "Silhouette (higher = better)",
               "Davies-Bouldin (lower = better)", "Calinski-Harabasz (higher = better)"],
    "Value": [FINAL_K,
              silhouette_score(X_scaled, final_labels),
              davies_bouldin_score(X_scaled, final_labels),
              calinski_harabasz_score(X_scaled, final_labels)],
}).style.hide(axis="index").format({"Value": "{:,.3f}"}))

A silhouette score of roughly 0.25-0.30 indicates **moderate** structure. This is typical for customer data, where behaviour forms a continuum rather than clearly separated islands. The segments are useful groupings, not natural "species".

---
## 9. Segment Profiling

Profiles use the **original (uncapped, untransformed)** values so the numbers are interpretable in business terms.

### 9.1 Size and revenue contribution

In [ ]:
segment_summary = customer_features.groupby("Cluster").agg(
    Customers=("CustomerID", "count"),
    Revenue=("Monetary", "sum"),
)
segment_summary["Customers (%)"] = segment_summary["Customers"] / segment_summary["Customers"].sum() * 100
segment_summary["Revenue (%)"] = segment_summary["Revenue"] / segment_summary["Revenue"].sum() * 100

display(segment_summary.style.format({
    "Customers": "{:,}", "Revenue": "£{:,.0f}",
    "Customers (%)": "{:.1f}%", "Revenue (%)": "{:.1f}%",
}))

### 9.2 Feature averages and medians per segment

Means are pulled up by large buyers inside a segment, so medians are shown alongside as a reality check.

In [ ]:
grouped = customer_features.groupby("Cluster")[feature_columns]

print("Mean per segment")
display(grouped.mean().round(2))

print("Median per segment")
display(grouped.median().round(2))

### 9.3 Heatmap: how each segment differs from the average customer

Values are the segment's mean on the *standardised* features: positive (dark blue) means above average, negative (red) means below average. For `Recency`, **high = long time since last purchase** (bad).

In [ ]:
profile_z = X_scaled_df.groupby(customer_features["Cluster"]).mean()

plt.figure(figsize=(13, 5))
sns.heatmap(profile_z, annot=True, fmt=".2f", cmap="RdBu", center=0,
            linewidths=0.5, cbar_kws={"label": "Std. deviations from average"})
plt.title("Segment Profile (standardised feature means)")
plt.xlabel("")
plt.ylabel("Cluster")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 9.4 Interpretation

Use the tables and heatmap above to name each segment. Cluster 0 is always the biggest spender. In the original run of this analysis (before some cleaning changes), four segments emerged that read roughly as follows. **Re-check against your own output**, since exact numbers shift slightly after re-running.

| Segment | Typical profile | Suggested action |
|---------|-----------------|------------------|
| **Champions / loyal high-value** | Bought very recently, ~11 orders, highest spend by far | Reward, early access, referral programmes; protect at all costs |
| **Regular mid-value** | Recent-ish, ~3 orders, moderate order value | Cross-sell, nudge towards a higher purchase frequency |
| **Lapsed big-basket, one-time** | Long since last purchase, ~1 order but with high order value | Win-back campaigns; these customers have shown willingness to spend |
| **Low-value lapsed** | Long inactive, ~1 small order | Low-cost reactivation only; do not over-invest |

**Limitations**

- Segments are based on a single year of data from one retailer, many of whose customers are probably wholesale buyers rather than individual consumers.
- Cluster quality is moderate (see 8.3); treat borderline customers as belonging to a spectrum.
- The revenue table in 9.1 is usually the most persuasive output: a small share of customers typically generates a large share of revenue.

---
## 10. Export Results

In [ ]:
output_cols = ["CustomerID", "Country", "Cluster"] + feature_columns
customer_features[output_cols].to_csv("customer_segments.csv", index=False)
print(f"Saved customer_segments.csv ({len(customer_features):,} customers)")

---
## References

- Dataset: Chen, D. (2015). *Online Retail*. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/352/online+retail
- Inspiration: [Customer Data Clustering (Kaggle, A. Chahardoli)](https://www.kaggle.com/code/alirezachahardoli/customer-data-clustering)
- Inspiration: [Customer Segmentation & Clustering (Kaggle, K. Kapoor)](https://www.kaggle.com/code/karnikakapoor/customer-segmentation-clustering)